In [4]:
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
from rasterio.transform import from_bounds
import numpy as np

# --- 1. Definizione dei percorsi ---
csv_path = r"C:\Users\Fra\OneDrive - Politecnico di Milano\File di Mattia Cinchetti - Thesis_onstoveM&F\OnStoveThesis\thesis\script_nostri\archived script galga\dataset_first_step\support_files\fob_per_kg.csv"
output_tif_path = r"C:\Users\Fra\OneDrive - Politecnico di Milano\File di Mattia Cinchetti - Thesis_onstoveM&F\repo_2jun\OnStoveThesis\thesis\script_nostri\dataset\fob_per_kg.tif"

# --- 2. Caricamento dei dati ---
df = pd.read_csv(csv_path)

# Assegna il nome 'iso3' alla prima colonna
df.columns.values[0] = 'iso3'

# Carica i confini dei paesi direttamente dalla fonte ufficiale
url_natural_earth = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"
world = gpd.read_file(url_natural_earth)

# --- 3. Unione e Filtraggio ---
# Nel dataset Natural Earth la colonna è 'ISO_A3' in maiuscolo
merged = world.merge(df, left_on='ISO_A3', right_on='iso3', how='inner')

# --- 4. Sostituzione dei valori -1 con la media dei confinanti ---
merged['fob_final'] = merged['fob_per_kg'].astype(float)
nazioni_mancanti = merged[merged['fob_final'] == -1].index

for idx in nazioni_mancanti:
    geometria_nazione = merged.loc[idx, 'geometry']
    confinanti = merged[merged.geometry.touches(geometria_nazione) | merged.geometry.intersects(geometria_nazione)]
    
    valori_validi = confinanti[(confinanti.index != idx) & (confinanti['fob_per_kg'] != -1)]['fob_per_kg']
    
    if not valori_validi.empty:
        media_confinanti = valori_validi.mean()
        merged.loc[idx, 'fob_final'] = media_confinanti
        print(f"Sostituito -1 per {merged.loc[idx, 'ISO_A3']} con {media_confinanti:.4f}")
    else:
        print(f"Attenzione: Nessun confinante valido con dati per {merged.loc[idx, 'ISO_A3']}. Il valore resta -1.")

# --- 5. Creazione del Raster tramite Rasterio ---
print("Generazione del raster in corso ad alta risoluzione...")

# Definizione della risoluzione spaziale (0.01 gradi)
res = 0.01

# Estrazione dell'estensione geografica totale
minx, miny, maxx, maxy = merged.total_bounds

# Calcolo delle dimensioni della matrice raster
width = int(np.ceil((maxx - minx) / res))
height = int(np.ceil((maxy - miny) / res))

# Creazione della trasformazione affine
transform = from_bounds(minx, miny, maxx, maxy, width, height)

# Preparazione generatore di tuple (geometria, valore_da_assegnare)
shapes = ((geom, value) for geom, value in zip(merged.geometry, merged['fob_final']))

# Esecuzione del rasterize
raster_array = rasterize(
    shapes=shapes,
    out_shape=(height, width),
    transform=transform,
    fill=-9999,
    dtype='float32'
)

# --- 6. Salvataggio del file GeoTIFF ---
with rasterio.open(
    output_tif_path,
    'w',
    driver='GTiff',
    height=height,
    width=width,
    count=1,
    dtype=raster_array.dtype,
    crs=merged.crs,
    transform=transform,
    nodata=-9999
) as dst:
    dst.write(raster_array, 1)

print(f"File salvato con successo in: {output_tif_path}")

Sostituito -1 per SLE con 1.1434
Sostituito -1 per AGO con 0.7905
Generazione del raster in corso ad alta risoluzione...
File salvato con successo in: C:\Users\Fra\OneDrive - Politecnico di Milano\File di Mattia Cinchetti - Thesis_onstoveM&F\repo_2jun\OnStoveThesis\thesis\script_nostri\dataset\fob_per_kg.tif
